In [1]:
from datasets import load_dataset
import polars as pl


/lfs/skampere2/0/animjha/CleanSlate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_dataset("unlearning-cleanslate/cleanslate_dataset", "default")["train"].to_polars()
df_olmo = load_dataset("unlearning-cleanslate/verif-olmo-3-1125-32b-w100-s10","default")["train"].to_polars()

In [3]:
IRIS_ID = "9bd7dd3ff9165c9ecce1313aed6a2915fd8f43ab"
NEVER_GONNA_ID = "b9bb555ab4913dfeaafb409555e8f543f5180fc6"

ACTIVE_ID = IRIS_ID
reference_text = df_olmo.filter(pl.col("song_id") == ACTIVE_ID)["reference_target"].item()


In [4]:
import requests
from tqdm.auto import tqdm

API_URL = "https://api.infini-gram-mini.io/"
# Available: v2_dclm_all, v2_piletrain, v2_pileval, v2_cc_2025-{05,08,13,18,21,26,30}
INDEX = "v2_dclm_all"

def igm_query(query: str, index: str = INDEX, query_type: str = "count") -> dict:
    r = requests.post(
        API_URL,
        json={"index": index, "query_type": query_type, "query": query},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

# sanity check
igm_query("University of Washington")

{'count': 3202538, 'latency': 540.2848720550537}

In [5]:
words = reference_text.split()
MAX_NGRAM = None  # set to e.g. 15 to cap the ngram length
ngrams = [
    {"ngram": " ".join(words[i:j]), "word_start": i, "word_end": j, "n_words": j - i}
    for i in range(len(words))
    for j in range(i + 2, (min(i + MAX_NGRAM, len(words)) if MAX_NGRAM else len(words)) + 1)
]
print(f"{len(ngrams)} substrings of >=2 words (from {len(words)} words)")

29890 substrings of >=2 words (from 245 words)


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def lookup(item: dict) -> dict:
    try:
        res = igm_query(item["ngram"])
        if "error" in res:
            return {**item, "count": None, "error": res["error"]}
        return {**item, "count": res.get("count"), "error": None}
    except Exception as e:
        print("error")
        return {**item, "count": None, "error": str(e)}

with ThreadPoolExecutor(max_workers=1) as ex:
    results = list(tqdm(ex.map(lookup, ngrams), total=len(ngrams)))

res_df = pl.DataFrame(results)
res_df.sort(["n_words", "count"], descending=[True, True])

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

MIN_CHARS = 7

def count_with_retry(q: str, max_attempts: int = 6) -> int:
    for attempt in range(max_attempts):
        try:
            res = igm_query(q)
            if "error" in res:
                raise RuntimeError(res["error"])
            return res["count"]
        except Exception:
            if attempt == max_attempts - 1:
                raise
            time.sleep(0.5 * (2 ** attempt))
    raise RuntimeError("unreachable")

def _longest_span_at(i: int) -> tuple[int, int, int] | None:
    text, n = reference_text, len(reference_text)
    if n - i < MIN_CHARS or count_with_retry(text[i:i+MIN_CHARS]) == 0:
        return None
    best, lo, hi = MIN_CHARS, MIN_CHARS + 1, n - i
    while lo <= hi:
        mid = (lo + hi) // 2
        if count_with_retry(text[i:i+mid]) > 0:
            best, lo = mid, mid + 1
        else:
            hi = mid - 1
    return i, i + best, count_with_retry(text[i:i+best])

with ThreadPoolExecutor(max_workers=16) as ex:
    raw = [r for r in tqdm(ex.map(_longest_span_at, range(len(reference_text))),
                            total=len(reference_text)) if r]

raw.sort(key=lambda s: (s[0], -(s[1] - s[0])))
spans, max_end = [], -1
for s in raw:
    if s[1] > max_end:
        spans.append(s); max_end = s[1]

spans_df = pl.DataFrame(
    [{"start": a, "end": b, "length": b - a, "count": c,
      "span_text": reference_text[a:b]} for a, b, c in spans]
).sort("length", descending=True)
spans_df